# Comparing two networks layer by layer with CKA and RSA

Do two networks trained on the same task learn the same representations, and at which depth? Two
measures answer this without matching units one to one. Linear centered kernel alignment (CKA)
compares the two layers' stimulus-by-stimulus similarity structure and is invariant to rotation and
scaling of the features. Representational similarity analysis (RSA) builds each layer's
representational dissimilarity matrix (RDM) over the same stimuli and correlates the two RDMs; it is
the measure neuroscience uses to compare models with brains.

Papers:

- Simon Kornblith, Mohammad Norouzi, Honglak Lee and Geoffrey Hinton (2019), "Similarity of Neural
  Network Representations Revisited", ICML 2019, https://arxiv.org/abs/1905.00414
- Nikolaus Kriegeskorte, Marieke Mur and Peter Bandettini (2008), "Representational similarity
  analysis: connecting the branches of systems neuroscience", Frontiers in Systems Neuroscience 2:4,
  https://doi.org/10.3389/neuro.06.004.2008

TorchLens records every layer of both networks with one `tl.trace` each, including functional
calls such as `F.relu` and `F.max_pool2d` that module hooks miss. `tl.stats.cka` and
`tl.repgeom.rdm` / `tl.repgeom.compare_rdms` then score every pair of layers.

The two models are tiny CNNs of different width (8 and 16 base channels) and different random
seeds, trained here on generated images so the notebook runs offline on a laptop CPU in seconds.
Each image holds one of four shapes in one of four quadrants; both networks learn to name the shape.
The same cells work for any two vision models over a shared stimulus set, for example two
torchvision classifiers on a folder of photographs, or a model and a brain recording.

In [ ]:
# recipe-test: tool=matplotlib expected=True
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F

import torchlens as tl


def make_shapes(n: int, seed: int) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Noisy 16x16 images of one of four shapes in one of four quadrants."""
    g = torch.Generator().manual_seed(seed)
    yy, xx = torch.meshgrid(torch.arange(7) - 3, torch.arange(7) - 3, indexing="ij")
    stamps = torch.stack(
        [
            (yy.abs().maximum(xx.abs()) == 3).float(),  # square outline
            (yy**2 + xx**2 <= 9).float(),  # disc
            ((yy == 0) | (xx == 0)).float(),  # plus
            (yy.abs() == xx.abs()).float(),  # cross
        ]
    )
    shape = torch.randint(0, 4, (n,), generator=g)
    top, left = torch.randint(0, 10, (2, n), generator=g)  # anywhere on the canvas
    images = 0.3 * torch.randn(n, 1, 16, 16, generator=g)
    for i in range(n):
        images[i, 0, top[i] : top[i] + 7, left[i] : left[i] + 7] += stamps[shape[i]]
    quadrant = 2 * (top >= 5) + (left >= 5)  # which quadrant holds the shape's centre
    return images, shape, quadrant


class TinyCNN(nn.Module):
    """Three conv stages, global average pooling, a linear shape classifier."""

    def __init__(self, width: int = 8) -> None:
        super().__init__()
        self.conv1 = nn.Conv2d(1, width, 3, padding=1)
        self.conv2 = nn.Conv2d(width, 2 * width, 3, padding=1)
        self.conv3 = nn.Conv2d(2 * width, 4 * width, 3, padding=1)
        self.head = nn.Linear(4 * width, 4)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = F.max_pool2d(F.relu(self.conv1(x)), 2)
        x = F.max_pool2d(F.relu(self.conv2(x)), 2)
        x = F.relu(self.conv3(x)).mean(dim=(2, 3))  # position is pooled away here
        return self.head(x)


def train(
    model: nn.Module, images: torch.Tensor, labels: torch.Tensor, epochs: int = 12
) -> nn.Module:
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    for _ in range(epochs):
        for batch in torch.randperm(len(images)).split(64):
            optimizer.zero_grad()
            F.cross_entropy(model(images[batch]), labels[batch]).backward()
            optimizer.step()
    return model.eval()


train_images, train_shape, _ = make_shapes(1000, seed=0)

torch.manual_seed(1)
narrow = train(TinyCNN(width=8), train_images, train_shape)
torch.manual_seed(2)
wide = train(TinyCNN(width=16), train_images, train_shape)

test_images, test_shape, _ = make_shapes(400, seed=1)
accuracies = [
    (m(test_images).argmax(1) == test_shape).float().mean().item() for m in (narrow, wide)
]
RESULT = min(accuracies) > 0.9

## 1. Record every layer of both networks on the same stimuli

In [ ]:
# recipe-test: tool=matplotlib expected=True
stimuli, _, _ = make_shapes(300, seed=3)


def layer_outputs(model: nn.Module) -> dict[str, torch.Tensor]:
    """Every layer's output from one trace, in execution order, one row per stimulus."""
    trace = tl.trace(model, stimuli, save="all")
    return {
        label: trace[label].out.detach().flatten(1)
        for label in trace.layer_labels
        if label not in trace.output_layers  # the output alias repeats the last layer
    }


narrow_layers, wide_layers = layer_outputs(narrow), layer_outputs(wide)
widths = {
    label: (narrow_layers[label].shape[1], wide_layers[label].shape[1]) for label in wide_layers
}
print(widths)  # the two networks' layers differ in width; CKA and RSA do not need them to match
RESULT = list(narrow_layers) == list(wide_layers)

## 2. Score every pair of layers

`tl.repgeom.rdm` builds each layer's correlation-distance RDM once; `tl.repgeom.compare_rdms`
correlates two RDMs over their upper triangles (Spearman, diagonal excluded) and `tl.stats.cka`
computes linear CKA on the activations. To score every layer of one network against a single target
(a brain region's responses, or one layer of another model) in one call, `tl.neuro.compare(layers,
target, analyses=("rsa", "cka"))` gives the same two numbers per layer.

In [ ]:
# recipe-test: tool=matplotlib expected=True
labels = list(narrow_layers)
rdms_narrow = [tl.repgeom.rdm(acts, "correlation") for acts in narrow_layers.values()]
rdms_wide = [tl.repgeom.rdm(acts, "correlation") for acts in wide_layers.values()]
cka = torch.tensor(
    [[tl.stats.cka(a, b) for b in wide_layers.values()] for a in narrow_layers.values()]
)
rsa = torch.tensor([[tl.repgeom.compare_rdms(a, b) for b in rdms_wide] for a in rdms_narrow])


def diagonal_contrast(matrix: torch.Tensor) -> float:
    """Mean score of corresponding layers minus mean score of all other pairs."""
    on = matrix.diagonal().mean()
    off = (matrix.sum() - matrix.diagonal().sum()) / (matrix.numel() - len(matrix))
    return (on - off).item()


print("CKA diagonal contrast", round(diagonal_contrast(cka), 3))
print("RSA diagonal contrast", round(diagonal_contrast(rsa), 3))
RESULT = diagonal_contrast(cka) > 0 and diagonal_contrast(rsa) > 0

## 3. Draw the two heatmaps

A bright diagonal means corresponding layers of the two networks carry the same representational
geometry even though their widths and initializations differ, the finding Kornblith et al. used to
validate CKA. The input row and column match exactly because both networks saw the same stimuli.

In [ ]:
# recipe-test: tool=matplotlib expected=True
fig, axes = plt.subplots(1, 2, figsize=(11, 4.8))
for ax, matrix, title in [(axes[0], cka, "linear CKA"), (axes[1], rsa, "RSA (Spearman of RDMs)")]:
    image = ax.imshow(matrix.numpy(), vmin=0, vmax=1, cmap="viridis")
    ax.set_xticks(range(len(labels)), labels, rotation=60, ha="right", fontsize=7)
    ax.set_yticks(range(len(labels)), labels, fontsize=7)
    ax.set_xlabel("wide network (16 base channels)")
    ax.set_ylabel("narrow network (8 base channels)")
    ax.set_title(title)
    fig.colorbar(image, ax=ax, fraction=0.046)
fig.tight_layout()
fig.savefig("cka_rsa_two_models.png", dpi=150)
RESULT = len(fig.axes) == 4